# CleanOps — YOLOv8n Model Evaluation & Error Analysis

**Project:** An AI-Powered Application for Waste Reporting & Smart Cleaning Management  
**Model:** YOLOv8n (`yolov8n_cleanops_best.pt`)  
**Evaluation:** Held-out TACO test split mapped to five CleanOps classes

This notebook documents the evaluation performed on the separate prepared test copy. It does not modify the original `TACO_clean` dataset or remap labels again.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
from collections import Counter
import yaml
import pandas as pd
from ultralytics import YOLO

DOWNLOADS = Path(r"C:\Users\m\Downloads")
ORIGINAL_DATA = DOWNLOADS / "TACO_clean"
EVAL_DATA = DOWNLOADS / "TACO_clean_eval"
MODEL_PATH = DOWNLOADS / "ai" / "models" / "yolov8n_cleanops_best.pt"
MAPPING_PATH = DOWNLOADS / "class_mapping.json"
DATA_YAML = EVAL_DATA / "data.yaml"
RESULTS_DIR = EVAL_DATA / "runs" / "test_evaluation"

print("Original dataset exists:", ORIGINAL_DATA.exists())
print("Prepared evaluation dataset exists:", EVAL_DATA.exists())
print("Model exists:", MODEL_PATH.exists())
print("Mapping file exists:", MAPPING_PATH.exists())
print("Evaluation YAML exists:", DATA_YAML.exists())


## 2. Verify model classes and evaluation configuration

In [ ]:
model = YOLO(str(MODEL_PATH))
model_names = list(model.names.values()) if isinstance(model.names, dict) else list(model.names)

with DATA_YAML.open("r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

yaml_names = list(data_cfg["names"].values()) if isinstance(data_cfg["names"], dict) else list(data_cfg["names"])
print("Model classes:", model_names)
print("YAML classes:", yaml_names)
assert model_names == yaml_names, "Model class order does not match evaluation YAML."
assert data_cfg.get("test") == "test/images", "The YAML must point the test split to test/images."


## 3. Verify the original-to-CleanOps class mapping

In [ ]:
with MAPPING_PATH.open("r", encoding="utf-8") as f:
    mapping_data = json.load(f)

mapping = {int(k): int(v) for k, v in mapping_data["taco_class_id_to_cleanops_id"].items()}
expected_mapping = {
    0: 3, 1: 0, 2: 0, 3: 2, 4: 3, 5: 1, 6: 4, 7: 0, 8: 0,
    9: 4, 10: 0, 11: 1, 12: 0, 13: 0, 14: 3, 15: 0, 16: 0, 17: 4
}
assert mapping == expected_mapping, "Mapping differs from the mapping used for this evaluation."
print("Verified mapping entries:", len(mapping))
print("Mapping targets:", sorted(set(mapping.values())))


## 4. Audit the prepared test labels (read-only)

In [ ]:
labels_dir = EVAL_DATA / "test" / "labels"
label_files = sorted(labels_dir.glob("*.txt"))
class_counts = Counter()
malformed_rows = []
total_boxes = 0

for label_file in label_files:
    for line_no, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), start=1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            malformed_rows.append((label_file.name, line_no, line))
            continue
        try:
            class_id = int(parts[0])
            class_counts[class_id] += 1
            total_boxes += 1
        except ValueError:
            malformed_rows.append((label_file.name, line_no, line))

print("Test label files:", len(label_files))
print("Class ID counts:", dict(sorted(class_counts.items())))
print("Total instances:", total_boxes)
print("Malformed rows:", len(malformed_rows))

assert len(label_files) == 607
assert set(class_counts).issubset(set(range(5)))
assert total_boxes == 1821
assert not malformed_rows


## 5. Duplicate-label audit

In [ ]:
duplicate_lines = 0
for label_file in label_files:
    lines = [line.strip() for line in label_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    duplicate_lines += len(lines) - len(set(lines))

print("Exact duplicate boxes remaining:", duplicate_lines)
assert duplicate_lines == 0


## 6. Run held-out test evaluation

In [ ]:
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    project=str(EVAL_DATA / "runs"),
    name="test_evaluation",
    exist_ok=True
)

overall = {
    "Precision": float(metrics.box.mp),
    "Recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
}
pd.DataFrame([overall]).round(4)


## 7. Class-wise results

In [ ]:
class_rows = [
    {"Class": "Plastic", "Images": 461, "Instances": 943, "Precision": 0.669, "Recall": 0.632, "mAP50": 0.651, "mAP50-95": 0.469},
    {"Class": "Paper/Cardboard", "Images": 119, "Instances": 153, "Precision": 0.641, "Recall": 0.513, "mAP50": 0.543, "mAP50-95": 0.436},
    {"Class": "Glass", "Images": 12, "Instances": 44, "Precision": 0.213, "Recall": 0.227, "mAP50": 0.107, "mAP50-95": 0.0485},
    {"Class": "Metal", "Images": 93, "Instances": 156, "Precision": 0.708, "Recall": 0.596, "mAP50": 0.634, "mAP50-95": 0.532},
    {"Class": "Organic/Trash", "Images": 212, "Instances": 525, "Precision": 0.454, "Recall": 0.160, "mAP50": 0.172, "mAP50-95": 0.103},
]
class_results = pd.DataFrame(class_rows)
class_results


## 8. Confusion matrix and diagnostic plots

In [ ]:
plot_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
]
for filename in plot_files:
    path = RESULTS_DIR / filename
    print(filename, "->", path.exists(), f"({path.stat().st_size:,} bytes)" if path.exists() else "")


## 9. Error analysis

- **Metal** has the highest precision (70.8%), indicating the strongest precision among the five categories.
- **Plastic** has the highest mAP@50 (65.1%) and recall (63.2%).
- **Glass** has the lowest mAP@50 (10.7%) and only 44 test instances; its limited representation may contribute to weaker generalization.
- **Organic/Trash** has low recall (16.0%), indicating that many ground-truth objects in this category were missed.
- Overall recall (42.57%) suggests missed-object detection is a major improvement priority.

These class-wise metrics support the observations above. Specific confusion pairs should be identified by inspecting the generated confusion-matrix image rather than inferred from aggregate metrics alone.


## 10. Recommendations and conclusion

1. Increase the amount and visual diversity of Glass training data.
2. Review Organic/Trash examples and category mapping, then improve its training representation.
3. Use targeted augmentation for underrepresented or difficult categories.
4. Inspect false negatives and false positives in prediction samples.
5. Retrain and compare improvements using the same held-out test split.

**Overall test results:** Precision = 53.69%, Recall = 42.57%, mAP@50 = 42.14%, and mAP@50-95 = 31.75%.

The evaluation is complete. The reported values are test-set metrics and must not be confused with the model's earlier validation metrics.
